## Librerías

In [14]:
import torch
import transformers
from transformers import pipeline
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D

## Texto de entrada

In [2]:
# Leer texto desde archivo .txt
with open("texto.txt", "r", encoding="utf-8") as file:
    texto = file.read()

print("Texto cargado correctamente")
print(f"Longitud: {len(texto)} caracteres")

Texto cargado correctamente
Longitud: 1723 caracteres


## Traducción

In [4]:
print("Cargando modelo de traducción (español → inglés)...")
traductor = pipeline("translation_es_to_en", model="Helsinki-NLP/opus-mt-es-en", framework="pt")

# Dividimos el texto en fragmentos
fragmento1 = texto[:500]
fragmento2 = texto[500:1000]
fragmento3 = texto[1000:]

# Traducimos cada fragmento y lo concatenamos al anterior
traduccion = (
    traductor(fragmento1)[0]['translation_text'] + " " +
    traductor(fragmento2)[0]['translation_text'] + " " +
    traductor(fragmento3)[0]['translation_text']
)

print("\n TRADUCCIÓN AL INGLÉS")
print(traduccion)

Cargando modelo de traducción (español → inglés)...


c:\Users\Delante\Desktop\nlp-text-processing-pipeline\nlp\lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
c:\Users\Delante\Desktop\nlp-text-processing-pipeline\nlp\lib\site-packages\transformers\models\marian\tokenization_marian.py:197: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")



 TRADUCCIÓN AL INGLÉS
Technology has revolutionized virtually every aspect of our lives, from how we communicate to how we carry out our daily tasks, also changing the way we interact with the world around us. Its impact is evident in areas as diverse as education, health, transport, and trade, improving efficiency, accessibility and connectivity at levels never imagined before. However, despite the countless benefits that we experience technology also brings with it complex and significant challenges that we cannot ignore. One of the most important challenges is the digital divide, which exacerbates social and economic inequalities between those who have access to technologies and those who do not. This disparity limits the educational, employment and social opportunities of many people in different parts of the world, leaving entire communities behind in technological development. It has raised serious concerns about the privacy and security of personal data, as much of our daily ac

## Resumen

In [6]:
print("\nCargando modelo de resumen (BART)...")
pipe = pipeline("summarization", model="facebook/bart-large-cnn", framework="pt")

# Usamos el texto ya traducido al inglés
fragmento1 = traduccion[:500]
fragmento2 = traduccion[500:1000]
fragmento3 = traduccion[1000:]

# Generamos resumen
resumen = (
    pipe(fragmento1, max_length=80, min_length=30, do_sample=False)[0]['summary_text'] + " " +
    pipe(fragmento2, max_length=80, min_length=30, do_sample=False)[0]['summary_text'] + " " +
    pipe(fragmento3, max_length=80, min_length=30, do_sample=False)[0]['summary_text']
)

print("\n RESUMEN EM INGLÉS")
print(resumen)


Cargando modelo de resumen (BART)...

 RESUMEN EM INGLÉS
Technology has revolutionized virtually every aspect of our lives. Its impact is evident in areas as diverse as education, health, transport, and trade. However, despite the countless benefits that we experience technology also brings with it complex and significant challenges. The digital divide exacerbates social and economic inequalities. It limits educational, employment and social opportunities of many people. It has raised serious concerns about the privacy and security of personal data. Technology is a tool at the service of all humanity, rather than becoming a source of inequality or threat. transactions to social interactions, leave a digital trail that can be exploited. It is crucial to reflect carefully on these aspects to mitigate the risks.


## Análisis de sentimiento

In [9]:
sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", framework="pt")

result = sentiment(resumen)
print(result)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


[{'label': 'POSITIVE', 'score': 0.9872148036956787}]


In [ ]:
sentences = [
    # Positivas
    "La tecnología ha revolucionado nuestras vidas",
    "Mejora la educación, salud y transporte",
    "La tecnología mejora la eficiencia y conectividad",
    "La innovación tecnológica abre nuevas oportunidades",
    "Gracias a la tecnología podemos comunicarnos mejor",
    "Los avances digitales benefician a muchas personas",
    "La conectividad mejora el acceso a la información",
    "La tecnología es una herramienta poderosa para el progreso",

    # Negativas
    "Existe una brecha digital importante",
    "Hay problemas de privacidad y seguridad",
    "Puede generar desigualdad social",
    "Los datos personales pueden ser explotados",
    "La tecnología deja comunidades enteras atrás",
    "Las desigualdades económicas se agravan con la tecnología",
    "La privacidad de los usuarios está en riesgo",

    # Neutrales
    "Debemos reflexionar sobre su impacto",
    "Es necesario analizar los efectos tecnológicos",
    "La tecnología tiene ventajas y desventajas",
    "Su impacto depende del contexto y uso",
    "Se deben considerar los riesgos y beneficios",
]

labels = [2,2,2,2,2,2,2,2, 0,0,0,0,0,0,0, 1,1,1,1,1]

In [13]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(sentences)

vocab_size = len(tokenizer.word_index) + 1

encoded_docs = tokenizer.texts_to_sequences(sentences)
padded_sequence = pad_sequences(encoded_docs, maxlen=50)

In [26]:
embedding_vector_length = 32

model = Sequential()
model.add(Embedding(vocab_size, embedding_vector_length, input_length=50))
model.add(SpatialDropout1D(0.25))
model.add(LSTM(50, dropout=0.5, recurrent_dropout=0.5))
model.add(Dropout(0.2))
model.add(Dense(3, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.build(input_shape=(None, 50)) 
print(model.summary())

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 50, 32)         │         2,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_2             │ (None, 50, 32)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 50)             │        16,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           153 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,665 (76.82 KB)

 Trainable params: 19,665 (76.82 KB)

 Non-trainable params: 0 (0.00 B)

None


In [27]:
labels_array = np.array(labels)

history = model.fit(
    padded_sequence,
    labels_array,
    epochs=60,
    batch_size=4,
    verbose=1
)

Epoch 1/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2000 - loss: 1.1019
Epoch 2/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4500 - loss: 1.1002     
Epoch 3/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3500 - loss: 1.0926
Epoch 4/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4000 - loss: 1.0824
Epoch 5/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5500 - loss: 1.0780
Epoch 6/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5500 - loss: 1.0775
Epoch 7/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4000 - loss: 1.0822
Epoch 8/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5000 - loss: 1.0602
Epoch 9/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5000 - loss: 1.0612
Epoch 10/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6000 - loss: 1.0725
Epoch 11/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5000 - loss: 1.0395
Epoch 12/60
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4500 - loss: 1.03

In [28]:
def predict_sentiment_global(frases):
    label_map = {0: "Negativo", 1: "Neutral", 2: "Positivo"}

    scores = []
    print("ANÁLISIS FRASE POR FRASE")

    for frase in frases:
        tw = tokenizer.texts_to_sequences([frase])
        tw = pad_sequences(tw, maxlen=50)
        prediction = model.predict(tw, verbose=0)[0]
        idx = np.argmax(prediction)
        confidence = prediction[idx]
        scores.append(prediction)

        print(f"[{label_map[idx]:8s}] ({confidence:.0%}) → {frase[:60]}")

    # Valoración global
    scores_array = np.array(scores)
    media_global = scores_array.mean(axis=0)

    neg_score = media_global[0]
    neu_score = media_global[1]
    pos_score = media_global[2]

    idx_global = np.argmax(media_global)

    print("\n")
    print("VALORACIÓN GLOBAL DEL TEXTO")
    print(f"Negativo : {neg_score:.1%}")
    print(f"Neutral  : {neu_score:.1%}")
    print(f"Positivo : {pos_score:.1%}")
    print(f"\nSentimiento dominante en el texto: {label_map[idx_global]}")

predict_sentiment_global(sentences)

ANÁLISIS FRASE POR FRASE
[Positivo] (99%) → La tecnología ha revolucionado nuestras vidas
[Positivo] (99%) → Mejora la educación, salud y transporte
[Positivo] (100%) → La tecnología mejora la eficiencia y conectividad
[Positivo] (100%) → La innovación tecnológica abre nuevas oportunidades
[Positivo] (100%) → Gracias a la tecnología podemos comunicarnos mejor
[Positivo] (99%) → Los avances digitales benefician a muchas personas
[Positivo] (100%) → La conectividad mejora el acceso a la información
[Positivo] (100%) → La tecnología es una herramienta poderosa para el progreso
[Negativo] (98%) → Existe una brecha digital importante
[Negativo] (98%) → Hay problemas de privacidad y seguridad
[Negativo] (98%) → Puede generar desigualdad social
[Negativo] (98%) → Los datos personales pueden ser explotados
[Negativo] (98%) → La tecnología deja comunidades enteras atrás
[Negativo] (98%) → Las desigualdades económicas se agravan con la tecnología
[Negativo] (99%) → La privacidad de los usuarios 